# QSS Complete Workflow & Analysis

**All-in-One Notebook**: Data Curation → Feature Engineering → Model Training → Scanner → EDA

This notebook provides a complete interface for:
1. **Data Curation**: Download & validate price/fundamental data
2. **Feature Engineering**: Calculate technical & fundamental features
3. **Dataset Building**: Generate Dataset A (features) & Dataset B (labels)
4. **Model Training**: Train & evaluate XGBoost models
5. **Scanner**: Run ML-enhanced scanner
6. **EDA & Analysis**: Explore data, model results, and simulated trades

---

## Table of Contents

- [Setup](#Setup)
- [1. Data Curation](#1.-Data-Curation)
- [2. Feature Engineering](#2.-Feature-Engineering)
- [3. Dataset Building](#3.-Dataset-Building)
- [4. Model Training](#4.-Model-Training)
- [5. Scanner](#5.-Scanner)
- [6. EDA: Data Analysis](#6.-EDA:-Data-Analysis)
- [7. EDA: Model Results](#7.-EDA:-Model-Results)
- [8. EDA: Simulated Trades](#8.-EDA:-Simulated-Trades)
- [9. EDA: Feature Importance](#9.-EDA:-Feature-Importance)
- [10. EDA: Prediction Analysis](#10.-EDA:-Prediction-Analysis)

## Setup

In [ ]:
# Standard library
import sys
from pathlib import Path
import warnings
warnings.filterwarnings('ignore')

# Add project root to path
project_root = Path.cwd().parent
sys.path.insert(0, str(project_root))

# Data manipulation
import pandas as pd
import numpy as np
from datetime import datetime, timedelta

# Visualization
import matplotlib.pyplot as plt
import seaborn as sns

# Project modules
import config
from src.data_engine import DataRepository
from src.features import FeatureEngineer
from src.fundamental_engine import FundamentalEngine
from src.fundamental_merger import FundamentalMerger
from src.strategy import SEPAStrategy
from src.trade_simulator import TradeSimulator
from src.trading_config import TradingConfig
from src.dataset_merger import DatasetMerger, DatasetLoader
from src.model_preparation import TemporalSplitter, FeatureSelector
from src.train_model import SEPAModelTrainer
from src.evaluate_model import ModelEvaluator
from src.ml_scorer import MLScorer
from src.database import DatabaseManager

# Configure plotting
plt.style.use('seaborn-v0_8-darkgrid')
sns.set_palette("husl")
%matplotlib inline

# Display settings
pd.set_option('display.max_columns', None)
pd.set_option('display.max_rows', 100)
pd.set_option('display.float_format', lambda x: '%.4f' % x)

print("✅ Setup complete!")
print(f"Project root: {project_root}")
print(f"Data directory: {config.DATA_DIR}")
print(f"Database: {config.DB_PATH}")

---

## 1. Data Curation

Download and validate price & fundamental data

### 1.1 Update Universe

In [ ]:
# Initialize data repository
repo = DataRepository()

# Update universe (FMP screener or S&P 500)
tickers = repo.update_universe(source=config.UNIVERSE_SOURCE)

print(f"✅ Universe loaded: {len(tickers)} tickers")
print(f"Source: {config.UNIVERSE_SOURCE}")
print(f"\nSample tickers: {tickers[:10]}")

### 1.2 Download Price Data

In [ ]:
# Download price data for universe (or subset for testing)
USE_FULL_UNIVERSE = False  # Set to True for production

if USE_FULL_UNIVERSE:
    download_tickers = tickers
else:
    # Test with smaller set
    download_tickers = ['AAPL', 'MSFT', 'NVDA', 'GOOGL', 'AMZN', 'META', 'TSLA', 'JPM', 'V', 'WMT']
    print(f"⚠️ TEST MODE: Using {len(download_tickers)} tickers")

# Update cache
print(f"Downloading price data for {len(download_tickers)} tickers...")
results = repo.update_cache(download_tickers, force=False)

# Summary
success = sum(results.values())
failed = len(results) - success

print(f"\n✅ Price data downloaded")
print(f"Success: {success}, Failed: {failed}")

### 1.3 Download Fundamental Data (Optional)

In [ ]:
# Download fundamental data (requires FMP API key)
DOWNLOAD_FUNDAMENTALS = False  # Set to True if using ML

if DOWNLOAD_FUNDAMENTALS:
    fund_engine = FundamentalEngine()
    
    print(f"Downloading fundamental data for {len(download_tickers)} tickers...")
    fund_results = {}
    
    for ticker in download_tickers:
        try:
            df = fund_engine.get_fundamentals(ticker)
            fund_results[ticker] = df is not None and not df.empty
        except Exception as e:
            fund_results[ticker] = False
    
    success = sum(fund_results.values())
    print(f"\n✅ Fundamental data downloaded")
    print(f"Success: {success}/{len(download_tickers)}")
else:
    print("⏭️ Skipping fundamental data download (DOWNLOAD_FUNDAMENTALS=False)")

### 1.4 Data Quality Check

In [ ]:
# Check data quality
data_quality = []

for ticker in download_tickers[:10]:  # Check first 10
    try:
        df = repo.get_ticker_data(ticker)
        data_quality.append({
            'ticker': ticker,
            'rows': len(df),
            'start_date': df.index.min(),
            'end_date': df.index.max(),
            'missing_values': df.isnull().sum().sum(),
            'columns': len(df.columns)
        })
    except Exception as e:
        data_quality.append({
            'ticker': ticker,
            'rows': 0,
            'error': str(e)
        })

quality_df = pd.DataFrame(data_quality)
print("\nData Quality Check (first 10 tickers):")
quality_df

---

## 2. Feature Engineering

Calculate technical & fundamental features

### 2.1 Calculate Features for Single Ticker (Demo)

In [ ]:
# Demo: Calculate features for AAPL
demo_ticker = 'AAPL'

# Load price data
price_data = repo.get_ticker_data(demo_ticker)

# Load benchmark (SPY)
spy_data = repo.get_benchmark_data()

# Initialize feature engineer
fe = FeatureEngineer(benchmark_data=spy_data)

# Calculate lightweight features
df_light = fe.calculate_lightweight_features(price_data.copy())

# Calculate heavyweight features (alphas)
df_full = fe.calculate_heavyweight_features(df_light.copy(), demo_ticker)

print(f"✅ Features calculated for {demo_ticker}")
print(f"Columns: {len(df_full.columns)}")
print(f"\nFeature columns:")
print([col for col in df_full.columns if col not in price_data.columns])

# Display recent data
print(f"\nRecent data (last 5 days):")
df_full[['Close', 'SMA_50', 'SMA_150', 'SMA_200', 'RS', 'VCP_Ratio', 'alpha001']].tail()

### 2.2 Add Fundamental Features (Optional)

In [ ]:
# Add fundamental features
ADD_FUNDAMENTALS = False  # Set to True if using ML

if ADD_FUNDAMENTALS:
    merger = FundamentalMerger()
    df_with_fundamentals = merger.merge_ticker_data(demo_ticker, df_full.copy())
    
    print(f"✅ Fundamental features added")
    print(f"Total columns: {len(df_with_fundamentals.columns)}")
    
    # Show fundamental columns
    fund_cols = [col for col in df_with_fundamentals.columns if col not in df_full.columns]
    print(f"\nFundamental columns ({len(fund_cols)}):")
    print(fund_cols[:10])  # First 10
    
    df_demo = df_with_fundamentals
else:
    df_demo = df_full
    print("⏭️ Skipping fundamental features (ADD_FUNDAMENTALS=False)")

---

## 3. Dataset Building

Generate Dataset A (features) & Dataset B (labels)

### 3.1 Generate Dataset B (Trade Labels)

In [ ]:
# Configuration for Dataset B generation
GENERATE_DATASET_B = True  # Set to True to run simulation
START_DATE = '2023-01-01'
END_DATE = '2024-12-31'

if GENERATE_DATASET_B:
    print(f"Generating Dataset B: {START_DATE} to {END_DATE}")
    print(f"Using {len(download_tickers)} tickers")
    
    # Initialize components
    strategy = SEPAStrategy(benchmark_data=spy_data)
    trading_config = TradingConfig.default()
    
    # Run simulation
    simulator = TradeSimulator(
        data_repo=repo,
        strategy=strategy,
        feature_engine=fe,
        start_date=START_DATE,
        end_date=END_DATE,
        config=trading_config,
        tickers=download_tickers
    )
    
    print("\nRunning trade simulation...")
    dataset_b = simulator.run_simulation()
    
    # Get summary statistics
    stats = simulator.get_summary_statistics()
    
    print(f"\n✅ Dataset B generated")
    print(f"Total trades: {len(dataset_b)}")
    print(f"Win rate: {stats['win_rate']:.2%}")
    print(f"Avg return: {stats['avg_return_pct']:.2f}%")
    print(f"Label distribution: {dataset_b['label'].value_counts().to_dict()}")
    
    # Save to file
    dataset_b_path = config.DATA_DIR / 'ml' / f'dataset_b_{START_DATE}_{END_DATE}.parquet'
    dataset_b_path.parent.mkdir(parents=True, exist_ok=True)
    dataset_b.to_parquet(dataset_b_path)
    print(f"\nSaved to: {dataset_b_path}")
else:
    print("⏭️ Skipping Dataset B generation (GENERATE_DATASET_B=False)")
    # Load existing Dataset B
    dataset_b_path = config.DATA_DIR / 'ml' / 'dataset_b.parquet'
    if dataset_b_path.exists():
        dataset_b = pd.read_parquet(dataset_b_path)
        print(f"Loaded existing Dataset B: {len(dataset_b)} trades")
    else:
        dataset_b = None
        print("⚠️ No existing Dataset B found")

### 3.2 Generate Dataset A (Feature Snapshots)

In [ ]:
# Configuration for Dataset A generation
GENERATE_DATASET_A = True  # Set to True to calculate features

if GENERATE_DATASET_A and dataset_b is not None:
    print("Generating Dataset A (feature snapshots)")
    
    # Get unique tickers from Dataset B
    dataset_a_tickers = dataset_b['ticker'].unique().tolist()
    print(f"Tickers from Dataset B: {len(dataset_a_tickers)}")
    
    # Calculate features for each ticker
    dataset_a_list = []
    
    for i, ticker in enumerate(dataset_a_tickers, 1):
        try:
            # Load price data
            df = repo.get_ticker_data(ticker)
            
            # Calculate features
            df_features = fe.calculate_lightweight_features(df.copy())
            df_features = fe.calculate_heavyweight_features(df_features, ticker)
            
            # Add fundamentals if enabled
            if ADD_FUNDAMENTALS:
                df_features = merger.merge_ticker_data(ticker, df_features)
            
            # Add ticker column
            df_features['ticker'] = ticker
            df_features['date'] = df_features.index
            
            dataset_a_list.append(df_features)
            
            if i % 10 == 0:
                print(f"Processed {i}/{len(dataset_a_tickers)} tickers")
                
        except Exception as e:
            print(f"Error processing {ticker}: {e}")
    
    # Combine all tickers
    dataset_a = pd.concat(dataset_a_list, ignore_index=True)
    
    print(f"\n✅ Dataset A generated")
    print(f"Rows: {len(dataset_a)}")
    print(f"Columns: {len(dataset_a.columns)}")
    print(f"Date range: {dataset_a['date'].min()} to {dataset_a['date'].max()}")
    
    # Save to file
    dataset_a_path = config.DATA_DIR / 'ml' / f'dataset_a_{START_DATE}_{END_DATE}.parquet'
    dataset_a.to_parquet(dataset_a_path)
    print(f"\nSaved to: {dataset_a_path}")
else:
    print("⏭️ Skipping Dataset A generation")
    # Load existing Dataset A
    dataset_a_path = config.DATA_DIR / 'ml' / 'dataset_a.parquet'
    if dataset_a_path.exists():
        dataset_a = pd.read_parquet(dataset_a_path)
        print(f"Loaded existing Dataset A: {len(dataset_a)} rows")
    else:
        dataset_a = None
        print("⚠️ No existing Dataset A found")

### 3.3 Merge Dataset A + B

In [ ]:
# Merge datasets
if dataset_a is not None and dataset_b is not None:
    print("Merging Dataset A + B")
    
    # Save temporarily if not already saved
    temp_a = config.DATA_DIR / 'ml' / 'temp_dataset_a.parquet'
    temp_b = config.DATA_DIR / 'ml' / 'temp_dataset_b.parquet'
    
    dataset_a.to_parquet(temp_a)
    dataset_b.to_parquet(temp_b)
    
    # Use DatasetMerger
    loader = DatasetLoader()
    df_a = loader.load_dataset_a(temp_a)
    df_b = loader.load_dataset_b(temp_b)
    
    merger = DatasetMerger(df_a, df_b)
    merged_dataset = merger.merge(strategy='left')
    
    # Get merge statistics
    merge_stats = merger.get_merge_statistics()
    
    print(f"\n✅ Datasets merged")
    print(f"Merged rows: {len(merged_dataset)}")
    print(f"Match rate: {merge_stats['match_rate']:.2%}")
    print(f"Missing snapshots: {merge_stats['missing_snapshots']}")
    
    # Save merged dataset
    merged_path = config.DATA_DIR / 'ml' / 'training_dataset_final.parquet'
    merged_dataset.to_parquet(merged_path)
    print(f"\nSaved to: {merged_path}")
else:
    print("⚠️ Cannot merge: Dataset A or B missing")
    merged_dataset = None

---

## 4. Model Training

Train & evaluate XGBoost models

### 4.1 Prepare Training Data

In [ ]:
# Load merged dataset if not already in memory
if merged_dataset is None:
    merged_path = config.DATA_DIR / 'ml' / 'training_dataset_final.parquet'
    if merged_path.exists():
        merged_dataset = pd.read_parquet(merged_path)
        print(f"Loaded merged dataset: {len(merged_dataset)} rows")
    else:
        print("⚠️ No merged dataset found. Run Dataset Building section first.")

if merged_dataset is not None:
    # Prepare features and labels
    exclude_cols = ['ticker', 'entry_date', 'exit_date', 'entry_price', 'exit_price', 
                    'return_pct', 'days_held', 'exit_reason', 'label']
    
    feature_cols = [col for col in merged_dataset.columns if col not in exclude_cols]
    
    X = merged_dataset[feature_cols].copy()
    y = merged_dataset['label'].copy()
    
    print(f"\nFeature matrix: {X.shape}")
    print(f"Label vector: {y.shape}")
    print(f"Label distribution: {y.value_counts().to_dict()}")
    print(f"Class imbalance ratio: {y.value_counts()[0] / y.value_counts()[1]:.2f}:1")

### 4.2 Temporal Split & Feature Selection

In [ ]:
if merged_dataset is not None:
    # Create temporal splits
    splitter = TemporalSplitter(purge_gap_days=60)
    folds = splitter.create_folds(merged_dataset, date_column='entry_date')
    
    print(f"✅ Created {len(folds)} temporal folds")
    for i, fold in enumerate(folds):
        print(f"\nFold {i+1}:")
        print(f"  Train: {fold['train_start']} to {fold['train_end']} ({fold['train_size']} samples)")
        print(f"  Test:  {fold['test_start']} to {fold['test_end']} ({fold['test_size']} samples)")
    
    # Use first fold for demo
    X_train, X_test, y_train, y_test = splitter.get_fold_data(merged_dataset, fold_idx=0)
    
    print(f"\nFold 1 data:")
    print(f"Train: {X_train.shape}, Test: {X_test.shape}")
    
    # Feature selection
    print(f"\nPerforming feature selection...")
    selector = FeatureSelector(correlation_threshold=0.95)
    X_train_selected = selector.fit_transform(X_train, y_train)
    X_test_selected = selector.transform(X_test)
    
    selected_features = selector.get_selected_features()
    
    print(f"\n✅ Feature selection complete")
    print(f"Features: {X_train.shape[1]} → {len(selected_features)}")
    print(f"Removed: {X_train.shape[1] - len(selected_features)} features")

### 4.3 Train Model

In [ ]:
if merged_dataset is not None:
    # Initialize trainer
    trainer = SEPAModelTrainer(max_depth=3, learning_rate=0.01)
    
    # Option 1: Quick training (default params)
    print("Training model (default params)...")
    trainer.train_baseline(X_train_selected, y_train, X_test_selected, y_test)
    
    # Option 2: Optimized training (uncomment to use Optuna)
    # print("Training model with hyperparameter optimization...")
    # best_params = trainer.optimize_hyperparameters(
    #     X_train_selected, y_train, X_test_selected, y_test, n_trials=30
    # )
    # print(f"\nBest params: {best_params}")
    
    print(f"\n✅ Model trained")
    
    # Save model
    model_path = project_root / 'models' / 'model_notebook.json'
    model_path.parent.mkdir(parents=True, exist_ok=True)
    
    metadata = {
        'fold': 1,
        'feature_names': selected_features,
        'training_date': datetime.now().isoformat(),
        'train_size': len(X_train_selected),
        'test_size': len(X_test_selected)
    }
    
    trainer.save_model(str(model_path), metadata=metadata)
    print(f"\nSaved to: {model_path}")

### 4.4 Evaluate Model

In [ ]:
if merged_dataset is not None and trainer.model is not None:
    # Initialize evaluator
    evaluator = ModelEvaluator(
        model=trainer.model,
        X_test=X_test_selected,
        y_test=y_test,
        fold_name='notebook_fold'
    )
    
    # Run comprehensive evaluation
    results = evaluator.evaluate_all()
    
    print("\n" + "="*60)
    print("MODEL EVALUATION RESULTS")
    print("="*60)
    
    print(f"\n📊 Classification Metrics:")
    print(f"  Accuracy:  {results['classification']['accuracy']:.4f}")
    print(f"  Precision: {results['classification']['precision']:.4f}")
    print(f"  Recall:    {results['classification']['recall']:.4f}")
    print(f"  F1 Score:  {results['classification']['f1_score']:.4f}")
    print(f"  ROC-AUC:   {results['classification']['roc_auc']:.4f}")
    
    print(f"\n🎯 Precision@k:")
    for k, precision in results['precision_at_k'].items():
        print(f"  Top {k}%: {precision:.4f}")
    
    print(f"\n💰 Trading Simulation (threshold=0.6):")
    sim = results['trading_simulation']
    print(f"  Win Rate:    {sim['win_rate']:.2%}")
    print(f"  Avg Return:  {sim['avg_return']:.2f}%")
    print(f"  Total Trades: {sim['total_trades']}")
    
    print(f"\n📈 vs Baseline (SEPA-only):")
    baseline = evaluator.compare_to_baseline(baseline_win_rate=0.097)
    print(f"  Baseline Win Rate: {baseline['baseline_win_rate']:.2%}")
    print(f"  ML Win Rate:       {baseline['ml_win_rate']:.2%}")
    print(f"  Improvement:       {baseline['precision_improvement']:.1%}")
    
    # Generate plots
    eval_dir = project_root / 'evaluation'
    eval_dir.mkdir(exist_ok=True)
    
    evaluator.plot_roc_curve(str(eval_dir / 'roc_notebook.png'))
    evaluator.plot_precision_recall_curve(str(eval_dir / 'pr_notebook.png'))
    evaluator.plot_feature_importance(str(eval_dir / 'importance_notebook.png'))
    
    print(f"\n✅ Evaluation plots saved to: {eval_dir}")

---

## 5. Scanner

Run ML-enhanced scanner

### 5.1 Run SEPA Scanner (Basic)

In [ ]:
# Scan for SEPA signals
scan_date = datetime.now().strftime('%Y-%m-%d')
# scan_date = '2024-11-28'  # Or use specific date

print(f"Scanning for SEPA signals: {scan_date}")
print(f"Universe: {len(download_tickers)} tickers")

# Load data for all tickers
ticker_data = {}
for ticker in download_tickers:
    try:
        df = repo.get_ticker_data(ticker)
        ticker_data[ticker] = df
    except:
        pass

print(f"Loaded data for {len(ticker_data)} tickers")

# Calculate features
enriched_data = fe.process_universe_batch(ticker_data)

# Scan for signals
scan_results = strategy.batch_scan_universe(enriched_data, pd.Timestamp(scan_date))

print(f"\n✅ SEPA scan complete")
print(f"Qualifying stocks: {len(scan_results['qualifying'])}")
print(f"New triggers: {len(scan_results['triggers'])}")

if scan_results['triggers']:
    print(f"\nTriggered signals:")
    for ticker in scan_results['triggers']:
        print(f"  {ticker}: RS={scan_results['triggers'][ticker]['rs']:.2f}, "
              f"Vol={scan_results['triggers'][ticker]['volume_ratio']:.2f}")

### 5.2 Run ML-Enhanced Scanner

In [ ]:
# ML-enhanced scanning
USE_ML = True  # Set to False for SEPA-only
ML_THRESHOLD = 0.6

if USE_ML and scan_results['triggers']:
    # Load model
    model_path = project_root / 'models' / 'model_fold_1.json'
    
    if not model_path.exists():
        model_path = project_root / 'models' / 'model_notebook.json'
    
    if model_path.exists():
        scorer = MLScorer(str(model_path))
        
        # Prepare candidate data
        candidates = []
        for ticker in scan_results['triggers']:
            df = enriched_data[ticker]
            
            # Get latest data point
            latest = df.loc[:scan_date].iloc[-1]
            latest['ticker'] = ticker
            candidates.append(latest)
        
        candidates_df = pd.DataFrame(candidates)
        
        # Add fundamentals if needed
        if ADD_FUNDAMENTALS:
            for ticker in candidates_df['ticker']:
                try:
                    df_temp = candidates_df[candidates_df['ticker'] == ticker].copy()
                    df_temp.index = pd.to_datetime([scan_date])
                    df_merged = merger.merge_ticker_data(ticker, df_temp)
                    candidates_df.loc[candidates_df['ticker'] == ticker] = df_merged.iloc[0]
                except:
                    pass
        
        # Score candidates
        probabilities, ranks = scorer.score_batch(candidates_df, ticker_column='ticker')
        
        # Filter by threshold
        filtered = scorer.filter_by_threshold(
            candidates_df, probabilities, ranks, threshold=ML_THRESHOLD
        )
        
        print(f"\n✅ ML scoring complete")
        print(f"Candidates scored: {len(candidates_df)}")
        print(f"Passed threshold (>{ML_THRESHOLD}): {len(filtered)}")
        
        if len(filtered) > 0:
            print(f"\n📊 ML-Filtered Signals (sorted by rank):")
            display_cols = ['ticker', 'ml_probability', 'ml_rank', 'Close', 'RS', 'VCP_Ratio']
            display_cols = [c for c in display_cols if c in filtered.columns]
            print(filtered[display_cols].sort_values('ml_rank'))
    else:
        print(f"⚠️ Model not found at {model_path}")
        print("Run Model Training section first or set USE_ML=False")
elif not USE_ML:
    print("⏭️ ML scoring disabled (USE_ML=False)")
else:
    print("⏭️ No triggers to score")

### 5.3 Update Database

In [ ]:
# Update buy list database
UPDATE_DATABASE = False  # Set to True to update database

if UPDATE_DATABASE and scan_results['triggers']:
    db = DatabaseManager()
    
    if USE_ML and 'filtered' in locals():
        # Add ML-filtered signals
        for _, row in filtered.iterrows():
            db.add_to_buy_list(
                ticker=row['ticker'],
                signal_date=scan_date,
                signal_price=row['Close'],
                current_price=row['Close'],
                ml_probability=row['ml_probability'],
                ml_rank=row['ml_rank'],
                ml_model_version=scorer.get_model_info()['model_version'],
                ml_score_date=scan_date,
                rs=row.get('RS'),
                volume_ratio=row.get('Vol_Ratio')
            )
    else:
        # Add SEPA-only signals
        for ticker in scan_results['triggers']:
            signal = scan_results['triggers'][ticker]
            db.add_to_buy_list(
                ticker=ticker,
                signal_date=scan_date,
                signal_price=signal['close'],
                current_price=signal['close'],
                rs=signal['rs'],
                volume_ratio=signal['volume_ratio']
            )
    
    print(f"\n✅ Database updated")
    
    # View buy list
    buy_list = db.get_buy_list(active_only=True)
    print(f"\nActive signals in database: {len(buy_list)}")
else:
    print("⏭️ Database update skipped (UPDATE_DATABASE=False)")

---

## 6. EDA: Data Analysis

Explore price data, features, and distributions

### 6.1 Price Data Overview

In [ ]:
# Analyze price data for a ticker
analysis_ticker = 'AAPL'
df_analysis = repo.get_ticker_data(analysis_ticker)

fig, axes = plt.subplots(3, 1, figsize=(15, 12))

# Price chart
axes[0].plot(df_analysis.index, df_analysis['Close'], label='Close', linewidth=1.5)
axes[0].set_title(f'{analysis_ticker} - Price Chart', fontsize=14, fontweight='bold')
axes[0].set_ylabel('Price ($)')
axes[0].grid(True, alpha=0.3)
axes[0].legend()

# Volume chart
axes[1].bar(df_analysis.index, df_analysis['Volume'], alpha=0.7)
axes[1].set_title(f'{analysis_ticker} - Volume', fontsize=14, fontweight='bold')
axes[1].set_ylabel('Volume')
axes[1].grid(True, alpha=0.3)

# Returns distribution
returns = df_analysis['Close'].pct_change()
axes[2].hist(returns.dropna(), bins=50, alpha=0.7, edgecolor='black')
axes[2].axvline(returns.mean(), color='red', linestyle='--', label=f'Mean: {returns.mean():.4f}')
axes[2].set_title(f'{analysis_ticker} - Daily Returns Distribution', fontsize=14, fontweight='bold')
axes[2].set_xlabel('Daily Return')
axes[2].set_ylabel('Frequency')
axes[2].legend()
axes[2].grid(True, alpha=0.3)

plt.tight_layout()
plt.show()

# Statistics
print(f"\n{analysis_ticker} Statistics:")
print(f"Date range: {df_analysis.index.min()} to {df_analysis.index.max()}")
print(f"Trading days: {len(df_analysis)}")
print(f"Current price: ${df_analysis['Close'].iloc[-1]:.2f}")
print(f"52-week high: ${df_analysis['Close'].iloc[-252:].max():.2f}")
print(f"52-week low: ${df_analysis['Close'].iloc[-252:].min():.2f}")
print(f"\nReturns:")
print(f"Mean daily return: {returns.mean():.4f}")
print(f"Volatility (daily): {returns.std():.4f}")
print(f"Sharpe ratio (annualized): {(returns.mean() / returns.std()) * np.sqrt(252):.2f}")

### 6.2 Feature Distributions

In [ ]:
# Analyze feature distributions
if df_demo is not None:
    key_features = ['SMA_50', 'SMA_150', 'SMA_200', 'RS', 'ATR', 'VCP_Ratio', 
                    'Consolidation_Width', 'Vol_Ratio']
    
    available_features = [f for f in key_features if f in df_demo.columns]
    
    if available_features:
        fig, axes = plt.subplots(2, 4, figsize=(20, 10))
        axes = axes.flatten()
        
        for i, feature in enumerate(available_features[:8]):
            data = df_demo[feature].dropna()
            axes[i].hist(data, bins=30, alpha=0.7, edgecolor='black')
            axes[i].axvline(data.median(), color='red', linestyle='--', 
                           label=f'Median: {data.median():.2f}')
            axes[i].set_title(feature, fontweight='bold')
            axes[i].legend()
            axes[i].grid(True, alpha=0.3)
        
        plt.tight_layout()
        plt.suptitle(f'{analysis_ticker} - Feature Distributions', 
                     fontsize=16, fontweight='bold', y=1.01)
        plt.show()
    else:
        print("⚠️ No features available for visualization")
else:
    print("⚠️ No feature data available. Run Feature Engineering section first.")

### 6.3 Feature Correlations

In [ ]:
# Feature correlation heatmap
if df_demo is not None and available_features:
    # Calculate correlation matrix
    corr_matrix = df_demo[available_features].corr()
    
    # Plot heatmap
    plt.figure(figsize=(12, 10))
    sns.heatmap(corr_matrix, annot=True, fmt='.2f', cmap='coolwarm', 
                center=0, square=True, linewidths=1)
    plt.title(f'{analysis_ticker} - Feature Correlation Matrix', 
              fontsize=14, fontweight='bold')
    plt.tight_layout()
    plt.show()
    
    # Find highly correlated pairs
    high_corr_pairs = []
    for i in range(len(corr_matrix.columns)):
        for j in range(i+1, len(corr_matrix.columns)):
            if abs(corr_matrix.iloc[i, j]) > 0.8:
                high_corr_pairs.append((
                    corr_matrix.columns[i], 
                    corr_matrix.columns[j], 
                    corr_matrix.iloc[i, j]
                ))
    
    if high_corr_pairs:
        print("\n⚠️ Highly correlated feature pairs (|r| > 0.8):")
        for feat1, feat2, corr in high_corr_pairs:
            print(f"  {feat1} ↔ {feat2}: {corr:.3f}")

---

## 7. EDA: Model Results

Analyze model predictions and performance

### 7.1 Prediction Distribution

In [ ]:
# Analyze prediction distribution
if merged_dataset is not None and trainer.model is not None:
    # Get predictions
    y_pred_proba = trainer.predict_proba(X_test_selected)
    
    fig, axes = plt.subplots(1, 2, figsize=(15, 5))
    
    # Overall distribution
    axes[0].hist(y_pred_proba, bins=50, alpha=0.7, edgecolor='black')
    axes[0].axvline(ML_THRESHOLD, color='red', linestyle='--', 
                   label=f'Threshold: {ML_THRESHOLD}')
    axes[0].set_title('Prediction Probability Distribution', fontsize=14, fontweight='bold')
    axes[0].set_xlabel('Probability')
    axes[0].set_ylabel('Frequency')
    axes[0].legend()
    axes[0].grid(True, alpha=0.3)
    
    # By actual label
    axes[1].hist(y_pred_proba[y_test == 1], bins=30, alpha=0.7, 
                label='Actual Winners', edgecolor='black')
    axes[1].hist(y_pred_proba[y_test == 0], bins=30, alpha=0.7, 
                label='Actual Losers', edgecolor='black')
    axes[1].axvline(ML_THRESHOLD, color='red', linestyle='--', 
                   label=f'Threshold: {ML_THRESHOLD}')
    axes[1].set_title('Prediction Distribution by Actual Label', fontsize=14, fontweight='bold')
    axes[1].set_xlabel('Probability')
    axes[1].set_ylabel('Frequency')
    axes[1].legend()
    axes[1].grid(True, alpha=0.3)
    
    plt.tight_layout()
    plt.show()
    
    # Statistics
    print("\nPrediction Statistics:")
    print(f"Mean probability: {y_pred_proba.mean():.4f}")
    print(f"Median probability: {np.median(y_pred_proba):.4f}")
    print(f"Std deviation: {y_pred_proba.std():.4f}")
    print(f"\nAbove threshold ({ML_THRESHOLD}): {(y_pred_proba >= ML_THRESHOLD).sum()} "
          f"({(y_pred_proba >= ML_THRESHOLD).mean():.1%})")
    
    # Calibration analysis
    print("\nCalibration Analysis:")
    for prob_range in [(0.0, 0.4), (0.4, 0.6), (0.6, 0.8), (0.8, 1.0)]:
        mask = (y_pred_proba >= prob_range[0]) & (y_pred_proba < prob_range[1])
        if mask.sum() > 0:
            actual_rate = y_test[mask].mean()
            print(f"  Prob {prob_range[0]:.1f}-{prob_range[1]:.1f}: "
                  f"{mask.sum()} samples, {actual_rate:.2%} actual win rate")

### 7.2 Confusion Matrix Analysis

In [ ]:
# Confusion matrix at different thresholds
if merged_dataset is not None and trainer.model is not None:
    from sklearn.metrics import confusion_matrix
    
    thresholds = [0.4, 0.5, 0.6, 0.7]
    
    fig, axes = plt.subplots(2, 2, figsize=(15, 12))
    axes = axes.flatten()
    
    for i, threshold in enumerate(thresholds):
        y_pred = (y_pred_proba >= threshold).astype(int)
        cm = confusion_matrix(y_test, y_pred)
        
        sns.heatmap(cm, annot=True, fmt='d', cmap='Blues', ax=axes[i],
                   xticklabels=['Predict Loss', 'Predict Win'],
                   yticklabels=['Actual Loss', 'Actual Win'])
        
        # Calculate metrics
        tn, fp, fn, tp = cm.ravel()
        precision = tp / (tp + fp) if (tp + fp) > 0 else 0
        recall = tp / (tp + fn) if (tp + fn) > 0 else 0
        
        axes[i].set_title(f'Threshold = {threshold}\n'
                         f'Precision: {precision:.2%}, Recall: {recall:.2%}',
                         fontweight='bold')
    
    plt.tight_layout()
    plt.suptitle('Confusion Matrices at Different Thresholds', 
                 fontsize=16, fontweight='bold', y=1.01)
    plt.show()

### 7.3 Precision-Recall Trade-off

In [ ]:
# Analyze precision-recall trade-off
if merged_dataset is not None and trainer.model is not None:
    from sklearn.metrics import precision_recall_curve
    
    precision, recall, thresholds = precision_recall_curve(y_test, y_pred_proba)
    
    fig, axes = plt.subplots(1, 2, figsize=(15, 5))
    
    # Precision-Recall curve
    axes[0].plot(recall, precision, linewidth=2)
    axes[0].set_title('Precision-Recall Curve', fontsize=14, fontweight='bold')
    axes[0].set_xlabel('Recall')
    axes[0].set_ylabel('Precision')
    axes[0].grid(True, alpha=0.3)
    
    # F1 score vs threshold
    f1_scores = 2 * (precision[:-1] * recall[:-1]) / (precision[:-1] + recall[:-1] + 1e-10)
    axes[1].plot(thresholds, f1_scores, linewidth=2)
    axes[1].axvline(ML_THRESHOLD, color='red', linestyle='--', 
                   label=f'Current threshold: {ML_THRESHOLD}')
    
    # Mark optimal F1
    optimal_idx = np.argmax(f1_scores)
    optimal_threshold = thresholds[optimal_idx]
    axes[1].axvline(optimal_threshold, color='green', linestyle='--', 
                   label=f'Optimal F1 threshold: {optimal_threshold:.2f}')
    
    axes[1].set_title('F1 Score vs Threshold', fontsize=14, fontweight='bold')
    axes[1].set_xlabel('Threshold')
    axes[1].set_ylabel('F1 Score')
    axes[1].legend()
    axes[1].grid(True, alpha=0.3)
    
    plt.tight_layout()
    plt.show()
    
    print(f"\nOptimal F1 Score: {f1_scores[optimal_idx]:.4f} at threshold {optimal_threshold:.2f}")
    print(f"Current threshold ({ML_THRESHOLD}): F1 = {f1_scores[np.argmin(np.abs(thresholds - ML_THRESHOLD))]:.4f}")

---

## 8. EDA: Simulated Trades

Analyze historical trade simulation results

### 8.1 Trade Distribution Analysis

In [ ]:
# Analyze Dataset B (simulated trades)
if dataset_b is not None:
    fig, axes = plt.subplots(2, 3, figsize=(20, 12))
    
    # Return distribution
    axes[0, 0].hist(dataset_b['return_pct'], bins=50, alpha=0.7, edgecolor='black')
    axes[0, 0].axvline(0, color='black', linestyle='--', alpha=0.5)
    axes[0, 0].axvline(dataset_b['return_pct'].mean(), color='red', linestyle='--', 
                      label=f'Mean: {dataset_b["return_pct"].mean():.2f}%')
    axes[0, 0].set_title('Return Distribution', fontweight='bold')
    axes[0, 0].set_xlabel('Return (%)')
    axes[0, 0].legend()
    axes[0, 0].grid(True, alpha=0.3)
    
    # Days held distribution
    axes[0, 1].hist(dataset_b['days_held'], bins=30, alpha=0.7, edgecolor='black')
    axes[0, 1].axvline(dataset_b['days_held'].mean(), color='red', linestyle='--',
                      label=f'Mean: {dataset_b["days_held"].mean():.1f} days')
    axes[0, 1].set_title('Days Held Distribution', fontweight='bold')
    axes[0, 1].set_xlabel('Days')
    axes[0, 1].legend()
    axes[0, 1].grid(True, alpha=0.3)
    
    # Exit reason
    exit_counts = dataset_b['exit_reason'].value_counts()
    axes[0, 2].bar(range(len(exit_counts)), exit_counts.values, alpha=0.7)
    axes[0, 2].set_xticks(range(len(exit_counts)))
    axes[0, 2].set_xticklabels(exit_counts.index, rotation=45)
    axes[0, 2].set_title('Exit Reasons', fontweight='bold')
    axes[0, 2].set_ylabel('Count')
    axes[0, 2].grid(True, alpha=0.3)
    
    # Return vs Days Held (scatter)
    winners = dataset_b[dataset_b['label'] == 1]
    losers = dataset_b[dataset_b['label'] == 0]
    
    axes[1, 0].scatter(losers['days_held'], losers['return_pct'], 
                      alpha=0.5, s=20, label='Losers')
    axes[1, 0].scatter(winners['days_held'], winners['return_pct'], 
                      alpha=0.5, s=20, label='Winners')
    axes[1, 0].axhline(0, color='black', linestyle='--', alpha=0.5)
    axes[1, 0].set_title('Return vs Days Held', fontweight='bold')
    axes[1, 0].set_xlabel('Days Held')
    axes[1, 0].set_ylabel('Return (%)')
    axes[1, 0].legend()
    axes[1, 0].grid(True, alpha=0.3)
    
    # Winners vs Losers distribution
    axes[1, 1].hist(winners['return_pct'], bins=30, alpha=0.7, 
                   label=f'Winners ({len(winners)})', edgecolor='black')
    axes[1, 1].hist(losers['return_pct'], bins=30, alpha=0.7, 
                   label=f'Losers ({len(losers)})', edgecolor='black')
    axes[1, 1].set_title('Winners vs Losers Returns', fontweight='bold')
    axes[1, 1].set_xlabel('Return (%)')
    axes[1, 1].legend()
    axes[1, 1].grid(True, alpha=0.3)
    
    # Label distribution pie chart
    label_counts = dataset_b['label'].value_counts()
    axes[1, 2].pie(label_counts.values, labels=['Losers', 'Winners'], 
                   autopct='%1.1f%%', startangle=90)
    axes[1, 2].set_title('Label Distribution', fontweight='bold')
    
    plt.tight_layout()
    plt.show()
    
    # Statistics
    print("\n" + "="*60)
    print("TRADE SIMULATION STATISTICS")
    print("="*60)
    print(f"\nTotal trades: {len(dataset_b)}")
    print(f"Winners: {len(winners)} ({len(winners)/len(dataset_b):.1%})")
    print(f"Losers: {len(losers)} ({len(losers)/len(dataset_b):.1%})")
    print(f"\nReturns:")
    print(f"  Mean: {dataset_b['return_pct'].mean():.2f}%")
    print(f"  Median: {dataset_b['return_pct'].median():.2f}%")
    print(f"  Std: {dataset_b['return_pct'].std():.2f}%")
    print(f"  Best: {dataset_b['return_pct'].max():.2f}%")
    print(f"  Worst: {dataset_b['return_pct'].min():.2f}%")
    print(f"\nHolding period:")
    print(f"  Mean: {dataset_b['days_held'].mean():.1f} days")
    print(f"  Median: {dataset_b['days_held'].median():.1f} days")
else:
    print("⚠️ No Dataset B available. Run Dataset Building section first.")

### 8.2 Top Winners & Losers

In [ ]:
# Analyze best and worst trades
if dataset_b is not None:
    print("\n" + "="*60)
    print("TOP 10 WINNERS")
    print("="*60)
    top_winners = dataset_b.nlargest(10, 'return_pct')
    display(top_winners[['ticker', 'entry_date', 'exit_date', 'return_pct', 
                         'days_held', 'exit_reason']])
    
    print("\n" + "="*60)
    print("TOP 10 LOSERS")
    print("="*60)
    top_losers = dataset_b.nsmallest(10, 'return_pct')
    display(top_losers[['ticker', 'entry_date', 'exit_date', 'return_pct', 
                        'days_held', 'exit_reason']])

### 8.3 Monthly Performance

In [ ]:
# Monthly performance analysis
if dataset_b is not None:
    # Add month column
    dataset_b['entry_month'] = pd.to_datetime(dataset_b['entry_date']).dt.to_period('M')
    
    # Group by month
    monthly_stats = dataset_b.groupby('entry_month').agg({
        'return_pct': ['count', 'mean', 'std'],
        'label': 'sum'
    })
    
    monthly_stats.columns = ['trades', 'avg_return', 'std_return', 'winners']
    monthly_stats['win_rate'] = monthly_stats['winners'] / monthly_stats['trades']
    
    # Plot
    fig, axes = plt.subplots(2, 1, figsize=(15, 10))
    
    # Monthly return
    x = range(len(monthly_stats))
    axes[0].bar(x, monthly_stats['avg_return'], alpha=0.7)
    axes[0].axhline(0, color='black', linestyle='--', alpha=0.5)
    axes[0].set_title('Average Return by Month', fontsize=14, fontweight='bold')
    axes[0].set_xticks(x)
    axes[0].set_xticklabels(monthly_stats.index.astype(str), rotation=45)
    axes[0].set_ylabel('Avg Return (%)')
    axes[0].grid(True, alpha=0.3)
    
    # Monthly win rate
    axes[1].bar(x, monthly_stats['win_rate'], alpha=0.7, color='green')
    axes[1].axhline(monthly_stats['win_rate'].mean(), color='red', linestyle='--',
                   label=f'Overall: {monthly_stats["win_rate"].mean():.1%}')
    axes[1].set_title('Win Rate by Month', fontsize=14, fontweight='bold')
    axes[1].set_xticks(x)
    axes[1].set_xticklabels(monthly_stats.index.astype(str), rotation=45)
    axes[1].set_ylabel('Win Rate')
    axes[1].legend()
    axes[1].grid(True, alpha=0.3)
    
    plt.tight_layout()
    plt.show()
    
    print("\nMonthly Statistics:")
    display(monthly_stats)

---

## 9. EDA: Feature Importance

Understand which features drive predictions

### 9.1 Top Feature Importance

In [ ]:
# Analyze feature importance
if merged_dataset is not None and trainer.model is not None:
    # Get feature importance
    importance = trainer.model.get_score(importance_type='gain')
    importance_df = pd.DataFrame({
        'feature': list(importance.keys()),
        'importance': list(importance.values())
    }).sort_values('importance', ascending=False)
    
    # Plot top 20
    top_n = 20
    top_features = importance_df.head(top_n)
    
    plt.figure(figsize=(12, 8))
    plt.barh(range(len(top_features)), top_features['importance'].values, alpha=0.7)
    plt.yticks(range(len(top_features)), top_features['feature'].values)
    plt.xlabel('Importance (Gain)')
    plt.title(f'Top {top_n} Most Important Features', fontsize=14, fontweight='bold')
    plt.gca().invert_yaxis()
    plt.grid(True, alpha=0.3, axis='x')
    plt.tight_layout()
    plt.show()
    
    print(f"\nTop {top_n} Features:")
    display(top_features)
    
    # Feature categories
    print("\nFeature Categories:")
    alpha_features = [f for f in importance_df['feature'] if 'alpha' in f.lower()]
    technical_features = [f for f in importance_df['feature'] 
                         if any(x in f.upper() for x in ['SMA', 'ATR', 'RS', 'VCP', 'VOL'])]
    fundamental_features = [f for f in importance_df['feature'] 
                           if any(x in f.lower() for x in ['revenue', 'eps', 'margin', 'ratio'])]
    
    print(f"Alpha factors: {len(alpha_features)}")
    print(f"Technical indicators: {len(technical_features)}")
    print(f"Fundamental metrics: {len(fundamental_features)}")
    print(f"Other: {len(importance_df) - len(alpha_features) - len(technical_features) - len(fundamental_features)}")

### 9.2 SHAP Feature Importance

In [ ]:
# SHAP-based feature importance (more accurate but slower)
CALCULATE_SHAP = False  # Set to True to calculate (may be slow)

if CALCULATE_SHAP and merged_dataset is not None and trainer.model is not None:
    print("Calculating SHAP importance (this may take a few minutes)...")
    
    # Use subset for speed
    sample_size = min(1000, len(X_test_selected))
    X_sample = X_test_selected.sample(sample_size, random_state=42)
    
    shap_importance = selector.calculate_feature_importance_shap(
        trainer.model, X_sample
    )
    
    # Plot top 20
    top_shap = shap_importance.head(20)
    
    plt.figure(figsize=(12, 8))
    plt.barh(range(len(top_shap)), top_shap['importance'].values, alpha=0.7)
    plt.yticks(range(len(top_shap)), top_shap['feature'].values)
    plt.xlabel('SHAP Importance (Mean Absolute)')
    plt.title('Top 20 Features by SHAP Values', fontsize=14, fontweight='bold')
    plt.gca().invert_yaxis()
    plt.grid(True, alpha=0.3, axis='x')
    plt.tight_layout()
    plt.show()
    
    print("\n✅ SHAP importance calculated")
    display(top_shap)
else:
    print("⏭️ SHAP calculation skipped (CALCULATE_SHAP=False)")
    print("Set CALCULATE_SHAP=True to enable (may be slow)")

---

## 10. EDA: Prediction Analysis

Analyze ML predictions on scanner results

### 10.1 Recent Predictions Log

In [ ]:
# Load and analyze prediction log
pred_log_path = config.DATA_DIR / 'predictions_log.parquet'

if pred_log_path.exists():
    pred_log = pd.read_parquet(pred_log_path)
    
    print(f"Prediction Log: {len(pred_log)} predictions")
    print(f"Date range: {pred_log['prediction_date'].min()} to {pred_log['prediction_date'].max()}")
    print(f"Unique tickers: {pred_log['ticker'].nunique()}")
    
    # Show recent predictions
    print("\nRecent Predictions (last 20):")
    recent = pred_log.sort_values('prediction_date', ascending=False).head(20)
    display(recent[['ticker', 'prediction_date', 'ml_probability', 'ml_rank', 
                    'actual_return_pct', 'actual_label']])
    
    # Predictions with outcomes
    completed = pred_log.dropna(subset=['actual_return_pct'])
    
    if len(completed) > 0:
        print(f"\nCompleted predictions: {len(completed)}")
        print(f"Win rate: {completed['actual_label'].mean():.1%}")
        print(f"Avg return: {completed['actual_return_pct'].mean():.2f}%")
        
        # Analyze by probability bins
        print("\nPerformance by Probability Bins:")
        completed['prob_bin'] = pd.cut(completed['ml_probability'], 
                                       bins=[0, 0.5, 0.6, 0.7, 0.8, 1.0],
                                       labels=['0-50%', '50-60%', '60-70%', '70-80%', '80-100%'])
        
        bin_analysis = completed.groupby('prob_bin').agg({
            'actual_label': ['count', 'mean'],
            'actual_return_pct': 'mean'
        })
        bin_analysis.columns = ['count', 'win_rate', 'avg_return']
        display(bin_analysis)
    else:
        print("\n⚠️ No completed predictions (outcomes not yet updated)")
else:
    print("⚠️ No prediction log found at:", pred_log_path)
    print("Run scanner with --use-ml to generate predictions")

### 10.2 Database Buy List Analysis

In [ ]:
# Analyze current buy list from database
db = DatabaseManager()
buy_list = db.get_buy_list(active_only=True)

if len(buy_list) > 0:
    print(f"\nActive Buy List: {len(buy_list)} signals")
    
    # Show top signals by ML rank
    if 'ml_rank' in buy_list.columns:
        print("\nTop 10 Signals by ML Rank:")
        top_signals = buy_list.sort_values('ml_rank').head(10)
        display(top_signals[['ticker', 'signal_date', 'ml_probability', 'ml_rank', 
                            'rs', 'volume_ratio', 'status']])
        
        # ML score distribution
        plt.figure(figsize=(12, 5))
        
        plt.subplot(1, 2, 1)
        plt.hist(buy_list['ml_probability'].dropna(), bins=20, alpha=0.7, edgecolor='black')
        plt.axvline(buy_list['ml_probability'].median(), color='red', linestyle='--',
                   label=f'Median: {buy_list["ml_probability"].median():.2f}')
        plt.title('ML Probability Distribution (Buy List)', fontweight='bold')
        plt.xlabel('Probability')
        plt.legend()
        plt.grid(True, alpha=0.3)
        
        plt.subplot(1, 2, 2)
        plt.scatter(buy_list['rs'], buy_list['ml_probability'], alpha=0.7)
        plt.title('ML Probability vs Relative Strength', fontweight='bold')
        plt.xlabel('RS')
        plt.ylabel('ML Probability')
        plt.grid(True, alpha=0.3)
        
        plt.tight_layout()
        plt.show()
    else:
        print("\n⚠️ No ML scores in buy list (run scanner with --use-ml)")
        display(buy_list[['ticker', 'signal_date', 'rs', 'volume_ratio', 'status']].head(10))
else:
    print("\n⚠️ Buy list is empty")
    print("Run scanner to populate buy list")

---

## Summary & Next Steps

**What you've built**:
- ✅ Data curation pipeline
- ✅ Feature engineering system
- ✅ ML training workflow
- ✅ Scanner integration
- ✅ Comprehensive EDA tools

**Next steps**:
1. **Iterate on features**: Add new technical/fundamental features based on importance analysis
2. **Tune threshold**: Adjust ML_THRESHOLD based on precision-recall analysis
3. **Monitor performance**: Track prediction log to validate model in production
4. **Retrain regularly**: Quarterly retraining with new data
5. **Build execution layer**: Integrate with broker API for automated trading

**Resources**:
- [USER_GUIDE.md](../USER_GUIDE.md) - Operational manual
- [docs/ARCHITECTURE.md](../docs/ARCHITECTURE.md) - System architecture
- [.claude/PROJECT_STANDARDS.md](../.claude/PROJECT_STANDARDS.md) - Development standards